In [11]:
import yfinance as yf
import pandas as pd
from pathlib import Path
import re

In [27]:
USA_DIR = Path("USA")
OUT_DIR = USA_DIR / "DatosHistoricos"
OUT_DIR.mkdir(parents=True, exist_ok=True)
xlsx_files = list(USA_DIR.glob("*.xlsx"))

tickers = [file.stem.strip() for file in xlsx_files if not file.name.startswith("~$")]

print(len(tickers))

41


In [28]:
# ticker = "AAPL"

df = yf.download(
    tickers=tickers,
    period="20y",
    interval="1wk",
    auto_adjust=True,
    group_by="ticker",
    progress=False
)

df = df.reset_index()


if isinstance(df.columns, pd.MultiIndex):
    new_cols = []
    for col in df.columns:
        if col[0] == "Date":
            new_cols.append(("Fecha", "")) 
        else:
            new_cols.append(col)
    
    df.columns = pd.MultiIndex.from_tuples(new_cols)

df.columns = pd.MultiIndex.from_tuples(
    [(ticker, 
      "Apertura" if field.lower() == "open" else
      "Máximo" if field.lower() == "high" else
      "Mínimo" if field.lower() == "low" else
      "Cierre" if field.lower() == "close" else
      "Vol." if field.lower() == "volume" else field)
     for ticker, field in df.columns]
)

In [29]:
##exportar a csv

tickers_en_df = [t for t in df.columns.get_level_values(0).unique() if t != "Fecha"]

cols_req = ["Apertura", "Máximo", "Mínimo", "Cierre", "Vol."]

for t in tickers_en_df:

    cols = [("Fecha", "")] + [(t, c) for c in cols_req if (t, c) in df.columns]
    one = df.loc[:, cols].copy()

    one.columns = ["Fecha"] + [c for c in cols_req if (t, c) in df.columns]

    if "Cierre" in one.columns:
        one = one.dropna(subset=["Cierre"])

    out_path = OUT_DIR / f"Datos históricos {t}.csv"
    one.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("Guardado:", out_path)

Guardado: USA\DatosHistoricos\Datos históricos NFLX.csv
Guardado: USA\DatosHistoricos\Datos históricos AMZN.csv
Guardado: USA\DatosHistoricos\Datos históricos MORN.csv
Guardado: USA\DatosHistoricos\Datos históricos DECK.csv
Guardado: USA\DatosHistoricos\Datos históricos ACN.csv
Guardado: USA\DatosHistoricos\Datos históricos FISV.csv
Guardado: USA\DatosHistoricos\Datos históricos FTNT.csv
Guardado: USA\DatosHistoricos\Datos históricos AMD.csv
Guardado: USA\DatosHistoricos\Datos históricos MMC.csv
Guardado: USA\DatosHistoricos\Datos históricos META.csv
Guardado: USA\DatosHistoricos\Datos históricos ABG.csv
Guardado: USA\DatosHistoricos\Datos históricos TSM.csv
Guardado: USA\DatosHistoricos\Datos históricos MNST.csv
Guardado: USA\DatosHistoricos\Datos históricos HD.csv
Guardado: USA\DatosHistoricos\Datos históricos MA.csv
Guardado: USA\DatosHistoricos\Datos históricos TER.csv
Guardado: USA\DatosHistoricos\Datos históricos BRO.csv
Guardado: USA\DatosHistoricos\Datos históricos CRM.csv
Guar